### **Simple RAG Implementation**

#### **Overview**

This notebook demonstrates a complete RAG pipeline using **LangChain 1.0+ with LCEL** (LangChain Expression Language).

**What is RAG?**

RAG combines retrieval of relevant documents with generation from a Large Language Model (LLM):
1. **Retrieval**: Find relevant information from a knowledge base
2. **Augmentation**: Add retrieved context to the prompt
3. **Generation**: LLM generates answers based on the context

**Pipeline Flow:**
```
PDF Documents → Load → Split into Chunks → Create Embeddings → Store in Vector DB
                                                                         ↓
User Query → Retrieve Similar Chunks → Combine with Query → LLM → Answer
```

**Components Used:**
- **Document Loader**: PyPDFLoader (for PDF processing)
- **Text Splitter**: RecursiveCharacterTextSplitter (smart chunking)
- **Embeddings**: OpenAI text-embedding-3-small (vector representations)
- **Vector Store**: FAISS (fast similarity search)
- **LLM**: OpenAI GPT-4-Turbo or GPT-3.5-Turbo
- **Chain Builder**: LCEL (LangChain Expression Language)

**LangChain 1.0+ Features:**
- ✅ Modern LCEL syntax with pipe operator `|`
- ✅ More readable and composable chains
- ✅ Better streaming support
- ✅ Type-safe operations

#### **2. Import Required Libraries**

Import all necessary modules with explanations of what each does.

##### **Verify Installation**

If you encounter import errors, run this cell first to check package versions:

!pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf python-dotenv tiktoken

In [1]:
# Check installed package versions
import sys
from importlib.metadata import version

try:
    import langchain
    print(f"✓ langchain: {langchain.__version__}")
except:
    print("✗ langchain not installed")

try:
    import langchain_core
    print(f"✓ langchain-core: {langchain_core.__version__}")
except:
    print("✗ langchain-core not installed - REQUIRED!")
    print("  Run: pip install langchain-core")

try:
    import langchain_openai
    print(f"✓ langchain-openai: {version('langchain-openai')}")
except:
    print("✗ langchain-openai not installed")

try:
    import langchain_community
    print(f"✓ langchain-community: {langchain_community.__version__}")
except:
    print("✗ langchain-community not installed")

print(f"\nPython version: {sys.version}")
print("\nIf any packages are missing, run:")
print("pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf python-dotenv tiktoken")

✓ langchain: 1.3.14
✓ langchain-core: 1.4.9
✓ langchain-openai: 1.3.5
✓ langchain-community: 0.4.2

Python version: 3.11.15 (main, Mar 20 2026, 00:32:44) [MSC v.1944 64 bit (AMD64)]

If any packages are missing, run:
pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf python-dotenv tiktoken


C:\Users\HP\AppData\Local\Temp\ipykernel_16920\4289264689.py:25: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [ ]:
# Standard library imports
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" ## sometimes due to FAISS and PyTorch, you may encounter a "libiomp5.dylib" error on MacOS. This line prevents that error.
from pathlib import Path

# Environment variable management - for secure API key handling
from dotenv import load_dotenv

# LangChain Document Loaders - for loading PDF documents
from langchain_community.document_loaders import PyPDFLoader

# LangChain Text Splitters - for breaking documents into manageable chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# OpenAI Integration - for embeddings and LLM
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Vector Store - FAISS for efficient similarity search
from langchain_community.vectorstores import FAISS

# LangChain Core Components
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("✓ All imports successful!")
print("✓ Compatible with LangChain 1.0+")

✓ All imports successful!
✓ Compatible with LangChain 1.0+


#### **3. Environment Configuration**

**Setting up OpenAI API Key**

You have two options:
1. **Recommended**: Create a `.env` file with `OPENAI_API_KEY=your_key_here`
2. **Alternative**: Set it directly in code (not recommended for production)

Get your API key from: https://platform.openai.com/api-keys

In [3]:
# Load environment variables from .env file
load_dotenv()

# Verify API key is loaded
if not os.getenv("OPENAI_API_KEY"):
    print("⚠️  WARNING: OPENAI_API_KEY not found!")
    print("Please set it in .env file or uncomment the line below:")
    # os.environ["OPENAI_API_KEY"] = "your_api_key_here"
else:
    print("✓ OpenAI API Key loaded successfully!")
    print(f"✓ Key starts with: {os.getenv('OPENAI_API_KEY')[:8]}...")

# Verify API key is loaded
if not os.getenv("GOOGLE_API_KEY"):
    print("⚠️  WARNING: GOOGLE_API_KEY not found!")
    print("Please set it in .env file or uncomment the line below:")
    # os.environ["OPENAI_API_KEY"] = "your_api_key_here"
else:
    print("✓ GOOGLE API Key loaded successfully!")
    print(f"✓ Key starts with: {os.getenv('GOOGLE_API_KEY')[:4]}...")

✓ OpenAI API Key loaded successfully!
✓ Key starts with: sk-proj-...
✓ GOOGLE API Key loaded successfully!
✓ Key starts with: AIza...


#### **4. Document Loading**

##### **4.1 Loading PDF Documents (PyPDFLoader)**

PyPDFLoader extracts text from PDF files page by page. Each page becomes a separate document with metadata (page number, source file).

**How it works:**
- Reads PDF files and extracts text content
- Preserves page numbers for source tracking
- Returns Document objects with `.page_content` and `.metadata`

In [6]:
pdf_path = "./sample_data\\PDFs\\attention.pdf" # Change this to your PDF file path


# Check if file exists
if not os.path.exists(pdf_path):
    print(f"⚠️  ERROR: File '{pdf_path}' not found!")
    print("Please update the pdf_path variable with your PDF file location.")
else:
    # Initialize the PDF loader
    pdf_loader = PyPDFLoader(pdf_path)
    
    # Load all pages from the PDF
    # Each page becomes a separate Document object
    documents = pdf_loader.load()
    
    # Display information about loaded documents
    print(f"✓ Loaded {len(documents)} pages from '{pdf_path}'")
    print(f"\n--- First Document Preview ---")
    print(f"Content (first 500 chars): {documents[0].page_content[:500]}...")
    print(f"\nMetadata: {documents[0].metadata}")
    print(f"\nTotal characters across all pages: {sum(len(doc.page_content) for doc in documents):,}")

✓ Loaded 15 pages from './sample_data\PDFs\attention.pdf'

--- First Document Preview ---
Content (first 500 chars): Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz ...

Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './sample_data\\PDFs

**If you have multiple PDF files, you can load them all at once:**

In [7]:
pdf_directory = "./sample_data\\PDFs"  # Directory containing your PDFs
all_documents = []

if os.path.exists(pdf_directory):
    pdf_files = list(Path(pdf_directory).glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")
    
    for pdf_file in pdf_files:
        multi_pdf_loader = PyPDFLoader(str(pdf_file))
        docs = multi_pdf_loader.load()
        all_documents.extend(docs)
        print(f"  ✓ Loaded {len(docs)} pages from {pdf_file.name}")
    
    print(f"\nTotal pages loaded: {len(all_documents)}")
    # documents = all_documents  # Use this for all the PDFs in the rest of the pipeline


Found 3 PDF files
  ✓ Loaded 15 pages from attention.pdf
  ✓ Loaded 19 pages from rag.pdf
  ✓ Loaded 21 pages from ragsurvey.pdf

Total pages loaded: 55


#### **5. Text Splitting**

**Why Split Documents?**
- LLMs have token limits (e.g., 4K, 8K, 128K tokens)
- Smaller chunks = more precise retrieval
- Balance: chunks must be large enough to contain meaningful context but small enough to be specific

**RecursiveCharacterTextSplitter**
This splitter tries to keep related text together by recursively splitting on:
1. Paragraphs (`\n\n`)
2. Lines (`\n`)
3. Sentences (`. `)
4. Words (` `)
5. Characters (as last resort)

**Parameters:**
- `chunk_size=1024`: Target size for each chunk (in characters)
- `chunk_overlap=128`: Overlap between chunks to maintain context continuity. We usually try to have an overlap of 10% -15% versus the chunk_size. 
- Overlap prevents important information from being split across chunks

In [8]:
# Initialize the text splitter with recommended settings
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,        # Maximum characters per chunk (roughly 200-250 tokens)
    chunk_overlap=128,      # Characters overlap between chunks (maintains context)
    length_function=len,    # Function to measure chunk length
    separators=["\n\n", "\n", " ", ""]  # Try to split on paragraphs first, then lines, etc. This is usually a chosen priority list. 
)

# Split the documents into chunks
# This creates smaller, manageable pieces while preserving semantic meaning
chunks = text_splitter.split_documents(documents)

# Display splitting results
print(f"✓ Split {len(documents)} documents into {len(chunks)} chunks")
print(f"\nAverage chunk size: {sum(len(chunk.page_content) for chunk in chunks) / len(chunks):.0f} characters")

# Preview a few chunks
print(f"\n--- Chunk Examples ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1} (length: {len(chunk.page_content)} chars):")
    print(f"{chunk.page_content[:200]}...")
    print(f"Metadata: {chunk.metadata}")

✓ Split 15 documents into 49 chunks

Average chunk size: 874 characters

--- Chunk Examples ---

Chunk 1 (length: 986 chars):
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './sample_data\\PDFs\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}

Chunk 2 (length: 944 chars):
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being mo

#### **6. Creating Embeddings (Paid Operation)**

**What are Embeddings?**

Embeddings are vector representations of text that capture semantic meaning. Similar texts have similar vectors.

**Example**: 
- "dog" and "puppy" → similar vectors (close in vector space)
- "dog" and "spaceship" → different vectors (far apart)

**OpenAI text-embedding-3-small**
- **Dimensions**: 1536 (each text becomes a 1536-dimensional vector)
- **Cost**: $0.00002 per 1,000 tokens (very affordable)
- **Performance**: 62.3% on MTEB benchmark
- **Speed**: Fast and efficient

**Alternative**: `text-embedding-3-large` for higher quality (64.6% MTEB) at higher cost

**Below we have 2 Embedding sample codes to choose from for OpenAI and Gemini**

##### **6.1 OpenAI Embeddings**

In [ ]:
# Initialize OpenAI Embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",  # cost-effective embedding model
    # dimensions=1536 It is not necessary that if the dimension is higher the result will be better
    # Alternative: "text-embedding-3-large" for better quality
)

# Test the embeddings with a sample text
sample_text = "This is a test sentence to demonstrate embeddings."
sample_embedding = embeddings.embed_query(sample_text)

print(f"✓ Embeddings model initialized: text-embedding-3-small")
print(f"✓ Embedding dimension: {len(sample_embedding)}")
print(f"✓ Sample embedding (first 10 values): {sample_embedding[:10]}")
print(f"\nℹ️  Each chunk will be converted to a {len(sample_embedding)}-dimensional vector for similarity search")

##### **6.2 Gemini Embeddings**

In [ ]:
# Below is the code for Gemini Embeddings:

from langchain_google_genai import GoogleGenerativeAIEmbeddings
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", output_dimensionality=768) 

# Test the embeddings with a sample text
sample_text = "This is a test sentence to demonstrate embeddings."
sample_embedding = embedding.embed_query(sample_text)

print(f"✓ Embeddings model initialized: gemini-embedding-001")
print(f"✓ Embedding dimension: {len(sample_embedding)}")
print(f"✓ Sample embedding (first 10 values): {sample_embedding[:10]}")
print(f"✓ Each chunk will be converted to a {len(sample_embedding)}-dimensional vector for similarity search")

✓ Embeddings model initialized: gemini-embedding-001
✓ Embedding dimension: 768
✓ Sample embedding (first 10 values): [-0.029574525, 0.022307422, 0.0063085253, -0.083313696, -0.00023175914, -0.0035217889, 0.0065565994, 0.011158722, 0.008888665, -0.009683166]
✓ Each chunk will be converted to a 768-dimensional vector for similarity search


Gemini embedding models (Gemini Embedding 001) are better than OpenAI models (text-embedding-3-small, text-embedding-3-large) as the gemini models have a higher MTEB score (68 versus 62/64) but the input tokens (limitation of the amount of data you can make an embedding for) are very very less () in gemini versus the OpenAI (4x)

Gemini newer model (gemini-embedding-2-preview) have a similar input token limit but the performance is not yet sure yet . 
- https://ai.google.dev/gemini-api/docs/embeddings
- https://developers.openai.com/api/docs/guides/embeddings

#### **7. Creating Vector Store (FAISS)**

**What is a Vector Store?**

A vector store (or vector database) stores embeddings and enables fast similarity search.

**FAISS (Facebook AI Similarity Search)**
- **Fast**: Optimized for billion-scale vector search
- **Local**: Runs on your machine, no cloud dependency
- **Efficient**: Uses advanced indexing algorithms

**How Similarity Search Works:**
1. Convert query to embedding vector
2. Find vectors in the database most similar to query vector (using cosine similarity or Euclidean distance)
3. Return the corresponding text chunks

**This cell will:**
1. Convert all chunks to embeddings (may take a minute for large documents)
2. Build a FAISS index
3. Save to disk for future use

**FAISS-CPU Version Compatibility Issue**

If you encounter issues with `faiss-cpu` installation, try:

```bash
uv pip uninstall faiss-cpu
uv pip install faiss-cpu==1.12.0
```

Or for conda users:
```bash
conda install -c conda-forge faiss-cpu==1.12.0
```

In [ ]:
# Create FAISS vector store from document chunks
# This step converts each chunk to an embedding and stores it
print(f"Creating FAISS index from {len(chunks)} chunks...")
print("This may take a minute depending on the number of chunks...")

vectorstore = FAISS.from_documents(
    documents=chunks,      # Our split document chunks
    embedding=embedding   # OpenAI embedding model/Gemini embedding.
)

print(f"✓ FAISS vector store created successfully!")
print(f"✓ Indexed {len(chunks)} document chunks")

# Save the vector store to disk for later use
# This allows you to reload the index without re-processing documents
vectorstore_path = "./faiss_index"
vectorstore.save_local(vectorstore_path)
print(f"✓ Vector store saved to '{vectorstore_path}'")
print(f"\nℹ️  You can reload this index later using: FAISS.load_local('{vectorstore_path}', embeddings)")

Creating FAISS index from 49 chunks...
This may take a minute depending on the number of chunks...
✓ FAISS vector store created successfully!
✓ Indexed 49 document chunks
✓ Vector store saved to './faiss_index'

ℹ️  You can reload this index later using: FAISS.load_local('./faiss_index', embeddings)


##### **7.1 ChromaDB Vector Store (Optional)**

In [ ]:
#ChromaDB has better Python 3.13 support. Replace cells 21-24 with:

# Instead of FAISS, use ChromaDB
# from langchain_community.vectorstores import Chroma

# # Create ChromaDB vector store
# print(f"Creating ChromaDB from {len(chunks)} chunks...")
# vectorstore = Chroma.from_documents(
#     documents=chunks,
#     embedding=embeddings,
#     persist_directory="./chroma_db"
# )
# print("✓ ChromaDB vector store created!")

##### **7.2 Loading a Saved Vector Store (Optional)**

If you've already created a vector store, you can load it instead of recreating:

In [ ]:
#Uncomment to load an existing vector store instead of creating a new one
# vectorstore_path = "./faiss_index"
# vectorstore = FAISS.load_local(
#     vectorstore_path, 
#     embeddings,
#     allow_dangerous_deserialization=True  # Required for loading pickled data
# )
# print(f"✓ Loaded existing vector store from '{vectorstore_path}'")

#### **8. Retriever**

In [12]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",    # Use cosine similarity for search
    search_kwargs={"k": 4}        # Retrieve top 4 most relevant chunks
)

print("✓ Retriever configured successfully")
print(f"  - Search type: similarity")
print(f"  - Number of documents to retrieve (k): 4")

# Test the retriever with a sample query
# Note: In LangChain 1.0+, use .invoke() instead of .get_relevant_documents()
test_query = "What is the main topic of this document?"
retrieved_docs = retriever.invoke(test_query)  # LangChain 1.0+ method

print(f"\n--- Retriever Test ---")
print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} documents:")

for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print(f"  Content preview: {doc.page_content[:150]}...")
    print(f"  Metadata: {doc.metadata}")

✓ Retriever configured successfully
  - Search type: similarity
  - Number of documents to retrieve (k): 4

--- Retriever Test ---
Query: 'What is the main topic of this document?'
Retrieved 4 documents:

Document 1:
  Content preview: Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
...
  Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './sample_data\\PDFs\\attention.pdf', 'total_pages': 15, 'page': 13, 'page_label': '14'}

Document 2:
  Content preview: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely

#### **9. Configuring the Language Model (LLM)**

**LLM Selection**
The LLM generates the final answer based on retrieved context.

**Available Models:**
1. **gpt-4-turbo-2025-04-09**: Most capable, best quality, slower, more expensive
2. **gpt-4o**: Fast GPT-4 level performance, good balance
3. **gpt-3.5-turbo**: Fast and cheap, good for simpler queries

**Temperature:**
- **0**: Deterministic, focused answers (recommended for factual Q&A)
- **0.7**: More creative, varied responses
- **1.0**: Most creative, less predictable

**Max Tokens:**
Controls the maximum length of the generated response.

In [ ]:
# Initialize the ChatOpenAI model
llm = ChatOpenAI(
      model="gpt-4-turbo-2024-04-09",  # Choose your model
      # Alternative options:
      # model="gpt-4o",           # Faster GPT-4 performance, good 
      # balance,
      # model="gpt-3.5-turbo",    # Faster and cheaper option

      temperature=0,         # 0 = deterministic, factual responses (recommended for Q&A)
      max_tokens=2000,       # Maximum length of response
  )

print("✓ LLM configured successfully")
print(f"  - Model: gpt-4-turbo-2024-04-09")
print(f"  - Temperature: 0 (deterministic)")
print(f"  - Max tokens: 2000")

# Test the LLM with a simple query
test_response = llm.invoke("Say 'Hello, I am ready to answer questions!'")
print(f"\nLLM Test Response: {test_response.content}")

#   📝 Explanation of Parameters:

#   Model Selection:

#   # Option 1: Best quality (slower, more expensive)
#   llm = ChatOpenAI(model="gpt-4-turbo-2024-04-09")

#   # Option 2: Fast GPT-4 performance (balanced)
#   llm = ChatOpenAI(model="gpt-4o")

#   # Option 3: Fast and cheap (good for testing)
#   llm = ChatOpenAI(model="gpt-3.5-turbo")

#   Temperature:

#   temperature=0    # Deterministic, focused (best for factual Q&A)
#   temperature=0.7  # More creative, varied responses
#   temperature=1.0  # Most creative, less predictable

#   Max Tokens:

#   max_tokens=2000  # Controls maximum response length

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)
response = llm.invoke("What is LangChain? Tell in 1 sentence.")
print("Question : What is LangChain? Tell in 1 sentence.")
print(f"Response : {response.content}")

Question : What is LangChain? Tell in 1 sentence.
Response : LangChain is a framework designed to simplify the development of applications powered by large language models by providing tools to connect them with external data sources and other computational components.


#### **10. Creating the RAG Chain (LangChain 1.0+ LCEL)**

**What is a RAG Chain?**

The RAG chain combines retrieval and generation into a single workflow:
1. User asks a question
2. Retriever finds relevant documents
3. Documents are formatted as context
4. LLM generates answer using the context

**LangChain 1.0+ LCEL (LangChain Expression Language)**

LangChain 1.0+ uses LCEL, a declarative way to build chains using the pipe operator `|`.

**Benefits:**

- More intuitive and readable
- Better streaming support
- Easier to debug and modify
- Type-safe and composable

**Components:**

- **RunnablePassthrough**: Passes input through unchanged
- **Pipe operator (|)**: Chains components together
- **StrOutputParser**: Converts LLM output to string

**Retrieved Docs(Context from VDB) + User Query + Prompt -------> LLM -> Final Answer**

In [14]:
# Define the prompt template for the RAG system
# This tells the LLM how to use the retrieved context
system_prompt = (
    "You are a helpful assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer based on the context, say that you don't know. "
    "Keep the answer concise and accurate.\n\n"
    "Context: {context}\n\n"
    "Question: {question}"
)

# Create the prompt template
prompt = ChatPromptTemplate.from_template(system_prompt)

# Helper function to format documents
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain using LangChain 1.0+ LCEL (LangChain Expression Language)
# This uses the pipe operator (|) to chain components together
rag_chain = (
    {
        "context": retriever | format_docs,  # Retrieve docs and format them
        "question": RunnablePassthrough()      # Pass through the question
    }
    | prompt           # Format with prompt template
    | llm              # Generate answer with LLM
    | StrOutputParser() # Parse output to string
)

print("✓ RAG chain created successfully using LangChain 1.0+ LCEL!")
print("\nRAG Pipeline Flow:")
print("  1. User provides a query")
print("  2. Retriever finds top 4 relevant chunks")
print("  3. Chunks are formatted as context")
print("  4. Context + question are formatted with prompt template")
print("  5. LLM generates answer based on context")
print("  6. Answer is parsed and returned to user")

✓ RAG chain created successfully using LangChain 1.0+ LCEL!

RAG Pipeline Flow:
  1. User provides a query
  2. Retriever finds top 4 relevant chunks
  3. Chunks are formatted as context
  4. Context + question are formatted with prompt template
  5. LLM generates answer based on context
  6. Answer is parsed and returned to user


In [15]:
# Example Query 1: General question about the document
query1 = "What is the main topic or subject of this document?"

print(f"Query: {query1}")
print("\nProcessing...\n")

# With LangChain 1.0+, we invoke the chain with the question directly
answer = rag_chain.invoke(query1)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

# To see which documents were retrieved, we can call the retriever separately
print("\nSOURCE DOCUMENTS USED:")
print("=" * 80)
retrieved_docs = retriever.invoke(query1)
for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print(f"  Source: {doc.metadata}")
    print(f"  Content: {doc.page_content[:200]}...")
    print("-" * 80)

Query: What is the main topic or subject of this document?

Processing...

ANSWER:
The main topic of this document is the attention mechanism, specifically its visualizations and role in handling long-distance dependencies and anaphora resolution within neural networks.


SOURCE DOCUMENTS USED:

Document 1:
  Source: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './sample_data\\PDFs\\attention.pdf', 'total_pages': 15, 'page': 13, 'page_label': '14'}
  Content: Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
sh...
-----------------------------

In [16]:
# Example Query 2: Specific information extraction
query2 = "Can you summarize the key points from this document?"

print(f"Query: {query2}")
print("\nProcessing...\n")

answer = rag_chain.invoke(query2)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

Query: Can you summarize the key points from this document?

Processing...

ANSWER:
The document illustrates attention mechanisms in neural networks, specifically in encoder self-attention layer 5. It shows how these mechanisms can follow long-distance dependencies, such as connecting "making" to "more difficult," and how they are involved in anaphora resolution, as demonstrated with the word "its."



In [17]:
# Example Query 3: Your custom question
# Replace this with your own question!
custom_query = "What specific details are mentioned about attention mechanisms?"

print(f"Query: {custom_query}")
print("\nProcessing...\n")

answer = rag_chain.invoke(custom_query)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

Query: What specific details are mentioned about attention mechanisms?

Processing...

ANSWER:
The context provides the following specific details about attention mechanisms:

*   **Encoder-decoder attention:** Queries come from the previous decoder layer, and memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence.
*   **Encoder self-attention:** Keys, values, and queries all come from the output of the previous layer in the encoder. Each position can attend to all positions in the previous layer.
*   **Decoder self-attention:** Keys, values, and queries all come from the previous layer in the decoder. Each position can attend to all positions in the decoder up to and including that position. Leftward information flow is prevented to preserve the auto-regressive property.
*   **Attention functions:**
    *   **Dot-product (multiplicative) attention** is used, scaled by `1/√dk`.
    *   **A

In [18]:
# Example Query 3: Your custom question
# Replace this with your own question!
custom_query = "What are the applications of Attention Mechanism?"

print(f"Query: {custom_query}")
print("\nProcessing...\n")

response3 = rag_chain.invoke(custom_query)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(response3)
print("\n" + "=" * 80)

Query: What are the applications of Attention Mechanism?

Processing...

ANSWER:
Attention mechanisms have been successfully used in:
*   Reading comprehension
*   Abstractive summarization
*   Textual entailment
*   Learning task-independent sentence representations
*   Simple-language question answering
*   Language modeling tasks
*   Sequence modeling and transduction models



### FAISS Kernel Crash ISSUE

Testing retrieval with query: 'What is the main topic of this document?'
OMP: Error #15: Initializing libomp.dylib, but found libomp.dylib already initialized.
OMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://openmp.llvm.org/